# DSM Circle Manifold

Chapter 17: Score Function and Energy-Based Models · *Mathematics for AI and Machine Learning* by Fuheng Wu

Denoising score matching on the unit circle.

Companion code for the [interactive demo](https://math4ai.org/companion/chapter-17/dsm-circle/) on math4ai.org (sign-in required). Run the cell with Shift+Enter; PyTorch is preinstalled on Colab. Change the values at the top and run it again to explore.


In [1]:
import math

import torch

torch.manual_seed(7)

# The demo's default noise level: data on the unit circle
sigma = 0.2


def sample_clean(n):
    t = 2 * math.pi * torch.rand(n)
    return torch.stack([t.cos(), t.sin()], dim=1)


net = torch.nn.Sequential(torch.nn.Linear(2, 128), torch.nn.SiLU(),
                          torch.nn.Linear(128, 128), torch.nn.SiLU(), torch.nn.Linear(128, 2))
opt = torch.optim.Adam(net.parameters(), lr=2e-3)

# ── DSM: E || s(x + sigma eps) + eps / sigma ||^2 ───────────────────────────
for step in range(1, 2001):
    x = sample_clean(512)
    eps = torch.randn_like(x)
    loss = ((sigma * net(x + sigma * eps) + eps) ** 2).sum(1).mean()   # = sigma^2 * DSM loss
    opt.zero_grad()
    loss.backward()
    opt.step()
    if step in (1, 500, 2000):
        print(f"step {step:4d}: loss {loss.item():.4f}")

# The loss levels off above 0: each target is a noisy score, so even the true score
# cannot fit it exactly. Judge the result by where the learned field points.
# ── The learned score is radial: inward outside the circle, outward inside ──
with torch.no_grad():
    print("\n radius   radial part of the score (+ = outward)")
    ang = torch.linspace(0, 2 * math.pi, 9)[:-1]
    for radius in (0.5, 0.8, 1.0, 1.2, 1.5):
        pts = radius * torch.stack([ang.cos(), ang.sin()], 1)
        radial = (net(pts) * pts / radius).sum(1)
        print(f"  {radius:.1f}     {radial.mean().item():+.3f}")
    # Following the score (a few small steps) pulls noisy points onto the circle
    pts = sample_clean(1000) + sigma * torch.randn(1000, 2)
    before = (pts.norm(dim=1) - 1).abs().mean().item()
    for _ in range(20):
        pts = pts + 0.2 * sigma ** 2 * net(pts)
    print(f"\nmean | |x| - 1 | for noisy samples: {before:.3f} -> {(pts.norm(dim=1) - 1).abs().mean().item():.3f} after following the score")

step    1: loss 1.9074


step  500: loss 1.0712


step 2000: loss 1.1688

 radius   radial part of the score (+ = outward)
  0.5     +9.900
  0.8     +4.363
  1.0     -0.566
  1.2     -5.539
  1.5     -12.928

mean | |x| - 1 | for noisy samples: 0.161 -> 0.022 after following the score
